To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

This notebook launch JEPO trainer

### Installation

In [ ]:
%pip install antlr4-python3-runtime==4.11
%pip install sympy

### Unsloth


Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:

import sys

# Save original stdout
original_stdout = sys.stdout

from pathlib import Path
current_dir = Path('./')
import shutil


from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoProcessor, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"



model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto")

 # Check memory usage
if torch.cuda.is_available():
    memory_used = torch.cuda.memory_allocated() / 1024**3
    memory_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 GPU Memory Used: {memory_used:.2f} GB / {memory_total:.1f} GB ({memory_used/memory_total*100:.1f}%)")

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from trl.data_utils import apply_chat_template, is_conversational, maybe_apply_chat_template, prepare_multimodal_messages
from sympy import simplify, sympify, parsing
from sympy.parsing.latex import parse_latex

### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset
from sympy import simplify, sympify, parsing
from sympy.parsing.latex import parse_latex
import warnings
import numpy as np

# Ignore only DeprecationWarnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Ignore only RuntimeWarnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Load and prep dataset
SYSTEM_PROMPT = """
You are an AI that systematically works through problems step by step before delivering a final response. Your output must include a 'reasoning' section that contains your internal analysis, followed by an 'answer' section that provides only the final answer with no extra explanation.
Respond as follows:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    # Extract the answer from the LaTeX string using the format \boxed{answer}
    extracted = re.search(r"\\boxed{(.*)}", text)
    if extracted:
        #in case there are multiple matches
        # return the last match group, which is the content inside the \boxed{}
        return extracted.groups()[-1].strip()
    return None

def str_to_sympy(expr_str: str, raw: str):
    expr = []
    try:
        expr.append(parse_latex(expr_str))
    except Exception as e:
        print(f"Error converting {expr_str}, {raw} to sympy expression: {e}")

    try:
        expr.append(parsing.sympy_parser.parse_expr(expr_str))
    except Exception as e:
        print(f"Error converting {expr_str} to sympy expression: {e}")
    return expr

# uncomment middle messages for 1-shot prompting
def get_math_questions(split = "train") -> Dataset:
    data = load_dataset('qwedsacf/competition_math')[split] # type: ignore
    print(data.shape)
    
    # Improved data processing with proper error handling
    def process_sample(x):
        answer_text = extract_hash_answer(x['solution'])
        
        # Verify the extracted answer can be parsed
        verified_count = 0
        if answer_text:
            verified_count = len(str_to_sympy(answer_text, x['solution']))
        
        return {
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': x['problem']}
            ],
            'answer': answer_text,
            'raw': x['solution'],
            'verified': verified_count
        }
    data = data.map(process_sample) # type: ignore
    
    # Filter to keep only samples with successfully parsed answers
    initial_size = len(data)
    data = data.filter(lambda x: x['verified'] > 0 and x['answer'] is not None) # type: ignore
    final_size = len(data)
    print(f"Filtered {initial_size - final_size} samples with invalid/unparseable answers. Kept {final_size} samples.")
    
    return data # type: ignore

dataset = get_math_questions()
print(dataset.shape)


def synpy_simplify_answer_check(response: str, answer: str) -> bool:
    try:
        if response is None or answer is None:
            print("response or answer is None, cannot compare.")
            return False
        
        if response.strip() == answer.strip():
            print("Exact string match found.")
            return True

        str_to_sympy_response = str_to_sympy(response, response)
        str_to_sympy_answer = str_to_sympy(answer, answer)

        for r in str_to_sympy_response:
            for a in str_to_sympy_answer:
                try:
                    if simplify(r - a) == 0:
                        return True
                except:
                    pass
        return False
    except Exception as e:
        print(f"Error simplifying answer: {e}")
        return False




def jepo_strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    #NOTE: re.DOTALL is critical here, otherwise the (.*) will not match newlines and we won't be able to extract the reasoning and answer correctly
    pattern = re.compile(r"<reasoning>(.*)</reasoning>\s*<answer>(.*)</answer>", re.DOTALL) 
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.0 if match else -1.0 for match in matches]



#given a completion and an answer, fabricate a ground truth completion that has the same reasoning but the answer is replaced with the ground truth answer
def fabricate_ground_truth_completion(completion: str, answer: str) -> (bool, str, str):
    #NOTE: re.DOTALL is critical here, otherwise the (.*) will not match newlines and we won't be able to extract the reasoning and answer correctly
    pattern = re.compile(r"<reasoning>(.*)</reasoning>\s*<answer>(.*)</answer>", re.DOTALL) 
    match = re.match(pattern, completion)

    if not match:
        #print("Completion does not have the correct format, cannot fabricate ground truth.", f"Completion: {completion}, Answer: {answer}")
        return False, completion, f"{completion}<answer>\n{answer}\n</answer>"
    else:
        reasoning = match.groups()[0].strip()
        ground_truth_completion = f"<reasoning>\n{reasoning}\n</reasoning>\n<answer>\n{answer}\n</answer>"
        return True, f"<reasoning>\n{reasoning}\n</reasoning>", ground_truth_completion



# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if synpy_simplify_answer_check(r, a) else 0.0 for r, a in zip(extracted_responses, answer)]
'''
NOT using this
def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]
'''
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]
'''
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]
'''
def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

<>:176: SyntaxWarning: invalid escape sequence '\s'
<>:176: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_298016/1337997636.py:176: SyntaxWarning: invalid escape sequence '\s'
  pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"


<a name="Train"></a>
### Train the model

Now set up JEPO Trainer and all configurations!

In [ ]:
#NOTE: max prompt length is 1628
#NOTE: max solution length is 2419 
#NOTE 1628 + 2419 = 4047, we set max_seq_length to 4608 to have some buffer for special tokens and generation
#NOTE 11929 samples
#batch size is 4 for H100 GPU


max_prompt_length = 1664
max_seq_length = 4608

from trl import JEPOConfig, JEPOTrainer
temp_output_dir = current_dir / "../jepo_math_norm"
if os.path.exists(temp_output_dir):
        print(f"Removing existing temporary directory: {temp_output_dir}")
        shutil.rmtree(temp_output_dir)
# Ensure the output directory exists
os.makedirs(temp_output_dir, exist_ok=True)
training_args = JEPOConfig(
    beta = 0.001,
    learning_rate = 4e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4, # Increase if you have more GPU memory
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    #max_steps = 1,
    num_train_epochs = 3, # Set to 3 for a full training run
    
    save_steps = 1000,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = temp_output_dir,
)

In [ ]:
print(f"Training configuration: {training_args}")

In [ ]:
# get the token length of prompts
lengths = dataset.map(lambda x: {"len": len(tokenizer(" ".join([d['content'] for d in x['prompt']]), truncation=False)['input_ids']),
        'solution_len': len(tokenizer(x['solution'], truncation=False)['input_ids'])
})
print(lengths.shape)
print(max(lengths['len']))
print(max(lengths['solution_len']))

In [ ]:
dataset[0]

dataset_all = dataset
dataset_all.shuffle(seed=42)

#75% for training, 25% for evaluation
train_size = int(0.75 * len(dataset_all))

dataset = dataset_all.select(range(train_size))
eval_dataset = dataset_all.select(range(train_size, len(dataset_all)))
print(dataset.shape)

In [ ]:
import os  
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
trainer = JEPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        jepo_strict_format_reward_func,
    ],
    cot_func = fabricate_ground_truth_completion,
    args = training_args,
    train_dataset = dataset,
)
trainer.train()
if torch.cuda.is_available():
    memory_used = torch.cuda.memory_allocated() / 1024**3
    memory_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 GPU Memory Used: {memory_used:.2f} GB / {memory_total:.1f} GB ({memory_used/memory_total*100:.1f}%)")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/anaconda/envs/dp-eval/lib/python3.13/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
1,-2.696400
2,-1.145700
3,0.154800
4,-0.101000
5,-2.916200
6,-2.978900
7,-0.907400
8,-1.637400
9,-2.700700
10,0.092300


RuntimeError: [enforce fail at inline_container.cc:664] . unexpected pos 11819474816 vs 11819474712

In [ ]:
f.flush()

In [ ]:
import numpy as np  
  
# Given array  
data = np.array([0, 0, 0, -1])  
  
# Compute the mean and standard deviation  
mean = np.mean(data)  
std = np.std(data)  
  
# Compute the z-score normalization: (x - mean) / std  
z_scores = (data - mean) / (std + 1e-4)  # Adding a small value to avoid division by zero  
  
print("Original array:", data)  
print("Mean:", mean)  
print("Standard Deviation:", std)  
print("Z-score normalized array:", z_scores)  

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

# Assuming you have already loaded your model and tokenizer.  
# model = LlamaForCausalLM.from_pretrained("path/to/your/model")  
# tokenizer = ...  
  
input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device) 
  
# Use the standard generate method.  
outputs = model.generate(  
    input_ids,  
    max_new_tokens=1024,  
    do_sample=True,  
    temperature=1.0,  
    top_p=0.95,  
)  
  
# Decode the generated tokens.  
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)  
print(output_text)  


And now with the LoRA we just trained with GRPO - we first save the LoRA first!

Now we load the LoRA and test:

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!